# Covered Today:
1. Selecting LLMs for Code Generation: Python to C++ with Cursor
2. Selecting Frontier Models: GPT, Claude, Grok & Gemini for C++ Code Gen
3. Porting Python to C++ with GPT
4. AI Coding Showdown: GPT vs Claude vs Gemini vs Groq Performance

Before moving ahead to topic 1, Ed spoke about the following:
1. As an LLM engineer, one of the most important task is to find the business problem we are trying to solve. When someone approaches us for a solution, we need to ask them what business problem are you trying to solve. It should be something tangible. The reason is that AI has received quite the hype and everyone wants a pie of that, however, AI may not solve for everything. 
2. Also, as an AI engineer, a person wears two hats: Software engineer and a Data Scientist. As per Ed, he gets most queries from learners related to SE. He says, these are not the most important questions. Most important questions are the Data Science parts: What is the business issue, how do we measure what are we solving for and what data do we have and what data do we need.

On the basis of above, he gives the below 5 steps to apply an LLM to a commercial problem(this will be covered later also.)
1. Understand: the business problem and how to measure it
2. Prepare: Select the best candidate models
3. Select: The model we are going to select
4. Customize: By building things like RAG or fine tuning the LLM
5. Production-ise: Scale

### 1. Selecting LLMs for Code Generation: Python to C++ with Cursor

Our next task is to write a program, which converts a given python script into an efficient C++ script. To do this, we start by viewing the benchmarks to shortlist the models we will be using for this task. Today we work with only frontier close sourced models and tomorrow we proceed with open source.

I will note down the steps taken next for identification of the candidate models:
1. We start with Artificial Analysis: we move to coding benchmarks. 

    First is Live code bench and the top 5 closed source models as per this benchmark are:
    1. Gemini 3 Pro Preview (high): 91.7 %
    2. Gemini 3 Flash Preview (Reasoning): 90.8%
    3. GPT 5.2 Medium: 89.4%
    4. GPT 5.2 xHigh: 88.9%
    5. Claude Opus 4.5: 87.1%. 

    Now, we move ahead with sci coding and the scores are:
    1. Claude Fable 5.1 (max with fallback): 63.1%
    2. Claude Fable 5 (with fallback): 61%
    3. Claude Fable 5.1 (xhigh with fallback): 60.9%
    4. Gemini 3.7 Flash (medium): 59.8%
    5. Gemini 3.7 Flash (medium): 59.7%

2. Now we move to vellum and in vellum we select coding LLMs, under this, we have two tests, and below are the scores
    LiveCodeBench
    1. Gemini 3 Pro: 79.7%
    2. Claude Opus 4.6: 76%
    3. OpenAI o3-mini: 74.1%
    4. Claude Sonnet 4.6: 72%
    5. GPT-4.1: 52%

    SWE Bench
    1. GPT-5.6 Sol: 96.2%
    2. Claude Mythos 5: 95.5%
    3. Claude Fable 5: 95%
    4. GPT-5.6 Luna: 93%
    5. Claude Opus 4.8: 88%

3. In the next step, we go to: SEAL and it has many benchmarks, below are the details:
    SWE Atlas - Refactoring: evaluates a model's ability to restructure production code while preserving behavior, which is our use case also.
    1. GPT 6 Astra (Codex) xHigh
    2. Fable-5.1 (Claude Code) xHigh
    3. Fable-5 (Claude Code)
    4. Opus-4.7 (Claude Code)
    5. Opus 4.8 (Claude Code)

4. Next we move to livebench, here under coding, we have the below leaders:
    1. Claude Fable 5.1 Max Effort
    2. Claude Fable 5 Max Effort
    3. GPT-5.6 Sol Max Effort
    4. GPT-5.2 Codex
    5. GPT-5.6 Luna Max Effort

5. And finally, we check Agent arena, here for coding, we have the below leaders:
    1. Claude Fable 5.1 (Max)
    2. GPT 6 Astra (Max)
    3. Claude Opus 5 (High)
    4. Claude Opus 5 (Max)
    5. Claude Fable 5 (High)


On the basis of above data, the final list of candidates is:
1. Claude Fable 5.1 (Max)
2. GPT 6 Astra (Codex) xHigh
3. Gemini 3 Pro

Adding gemini 3 pro, since I want to include models from major labs. In addition to these, gemini-3.1-pro-preview(latest pro version) and Sarvam-105B to the list to check how these hold up. 

In [1]:
# we now start by importing basic libraries, like openai, env, os to check if all models are working or not

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

In [3]:
load_dotenv(override=True)
openai_api = os.getenv("OPENAI_API_KEY")
google_api = os.getenv("GOOGLE_API_KEY")
claude_api = os.getenv("ANTHROPIC_API_KEY")
sarvam_api = os.getenv("SARVAM_API")

google_baseurl = "https://generativelanguage.googleapis.com/v1beta/openai/"
claude_baseurl = "https://api.anthropic.com/v1/"
sarvam_baseurl = "https://api.sarvam.ai/v1/"

In [4]:
# lets start by creating a function, that checks if APIs are working fine and models are responding or not.

def ping_llm(api_key, model, base_url):
  """Pings any OpenAI-compatible LLM endpoint.

  Returns True if reachable and responding, False otherwise.
  """
  if not api_key:
    print(f"[{model}] SKIPPED: API key is missing.")
    return False

  try:
    client = OpenAI(api_key=api_key, base_url=base_url)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": "Respond only with: pong"}],
        temperature=0,
    )
    reply = response.choices[0].message.content.strip() #type: ignore
    print(f"[{model}] ONLINE -> {reply}")
    return "All APIs connected and working."
  except Exception as err:
    print(f"[{model}] FAILED -> {err}")
    return False

In [4]:
ping_llm(openai_api, "gpt-4o-mini", None)
ping_llm(google_api, "gemini-3.5-flash-lite", base_url=google_baseurl)
ping_llm(claude_api, "claude-haiku-4-5-20251001", base_url=claude_baseurl)
ping_llm(sarvam_api, "sarvam-105b", base_url=sarvam_baseurl)

[gpt-4o-mini] ONLINE -> pong
[gemini-3.5-flash-lite] ONLINE -> pong
[claude-haiku-4-5-20251001] ONLINE -> pong
[sarvam-105b] ONLINE -> pong


'All APIs connected and working.'

In [5]:
# Next is the system info that we extract from the system. This has been written wih an LLM and fetches the information from file mac_sysinfo.py. The code is an improvement over the course code, since, we have added caching information, and memory sizing info.
# Import the function from saved file
from mac_sysinfo import get_unified_m2_context
system_info = get_unified_m2_context(as_json=True)
# Verify the output
print(system_info)

{
  "os_environment": {
    "system": "Darwin",
    "release": "27.0.0",
    "architecture": "arm64",
    "rosetta2_translated": false,
    "target_triple": "arm64-apple-darwin27.0.0"
  },
  "hardware_constraints": {
    "cpu_brand": "Apple M2",
    "cores": {
      "physical": 8,
      "logical": 8
    },
    "compute_extensions": [
      "AdvSIMD",
      "AdvSIMD_HPFPCvt",
      "FEAT_BF16",
      "FEAT_DotProd",
      "FEAT_FHM",
      "FEAT_FP16",
      "FEAT_I8MM",
      "floatingpoint",
      "neon",
      "neon_fp16",
      "neon_hpfp"
    ],
    "memory": {
      "total_ram_bytes": 17179869184,
      "page_size_bytes": 16384
    },
    "cache": {
      "cache_line_size_bytes": 128,
      "l1_data_cache_bytes": 65536,
      "l2_cache_bytes": 4194304
    }
  },
  "toolchain": {
    "compilers": {
      "clang": "Apple clang version 21.0.0 (clang-2100.3.34.2)",
      "gcc": "Apple clang version 21.0.0 (clang-2100.3.34.2)"
    },
    "build_tools": {
      "make": "GNU Make 3.81",


In [6]:
# now since we have the system info with us, lets us move ahead with the python code as given in the course.

pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [7]:
# let's now run the code. Since, the code is in a str format, we use exec() to run this code.
exec(pi)

Result: 3.141592656089
Execution Time: 14.432308 seconds


In [8]:
# Well, as we can see, the result is right, we wanted 3.14... as the output and the time taken by the machine is 14 seconds. Now, we have a python figure. Next we move ahead to C++. But before we go ahead, let's write down what we need our remaining program to do, so that we can build a function around it.
# 1. It should take in the given python code, which is standard, so it can be hard coded and must give us the output in the same manner, the way we ran the python code. 
# 2. We should be able to pass in the model name and it should then run the entire chain of porting the python code, bringing back the c++ code, adding it to a file, compiling it and then running it. 
# 3. We can now break down the various steps defined above into smaller functions and compile all of it together.
# > 1. Create system prompt with system info + user prompt with the python code and return that as output > 2. Next function to be able to compile the c++ code into a file and execute it, and return the final results, > 3. one function we create to call the model > we pass in a model info dict, and it runs the model. 

# Lets first create the model dict. Also, to be cognizant of the tokens, since these are expensive models, we keep the effort to low first, and if the tests fail, we move one level up. Also, as per google, gemini-3-pro is not available anymore, so we simply move to 3.1 pro
models = {
    'gpt': {
        'model': 'gpt-6-astra',
        'api_key': openai_api,
        'base_url': None,
    },
    'claude': {
        'model': 'claude-fable-5-1',
        'api_key': claude_api,
        'base_url': claude_baseurl,
    },
    'gemini': {
        'model': 'gemini-3.1-pro-preview',
        'api_key': claude_api,
        'base_url': claude_baseurl,
    },
    'sarvam': {
        'model': 'sarvam-105b',
        'api_key': sarvam_api,
        'base_url': sarvam_baseurl,
    }

}

In [9]:
# let's now define the function to run each of these models:
def fetch_model(provider):
    provider_details = models.get(provider)
    model = provider_details.get('model')
    api_key = provider_details.get('api_key')
    base_url = provider_details.get('base_url')

    return model, api_key, base_url

GPT, CLAUDE, GEMINI, SARVAM = "gpt", "claude", "gemini", "sarvam"

In [40]:
# Now we create a create message function, which can be used to run any given python code, for any given sys info and any compile command depending upon the system. The prompt text has been taken from the course.

messages = []

def create_message(sys_info, code_str, compile_info):
    system_message = f"""
    Your task is to convert Python code into high performance C++ code.
    Respond only with C++ code. Do not provide any explanation other than occasional comments.
    The C++ response needs to produce an identical output in the fastest possible time. The code must be optimized to run on the specified system details mentioned below:
    {sys_info}
    """

    user_message = f"""
    Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
    The system information is:
    {sys_info}
    Make sure that your code uses maximum use of the hardware and executes the code in the fastest possible time. You should also make maximum use of the cores in the hardware, to create code that can run simultaneously. Note that code being super fast is our main objective. You can change the algorithm as well if needed, we are not looking for translation of the code, but a code that does exactly what the below code achieved. 
    Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
    {compile_info}
    Respond only with C++ code.
    Python code to port:

    ```python
    {code_str}
    ```
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    return messages

In [41]:
# compile_message = ["clang++", "-std=c++17", "-O3", "-DNDEBUG", "main.cpp", "-o", "main"]

# trying the compile code used by ed to check if the compilation also impacts the results?
compile_message = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]

In [42]:
# now, lets create couple of functions in which, one function extracts the c++ code from the LLM response and the other function creates the main.cpp file and writes the code to it. 

import re
from pathlib import Path

def extract_cpp(llm_output):
    """Pull raw C++ out of whatever the model returned."""
    # Grab the contents of the first ```...``` block, if there is one.
    match = re.search(r"```(?:cpp|c\+\+)?\s*(.*?)```", llm_output, flags=re.DOTALL | re.IGNORECASE)
    code = match.group(1) if match else llm_output   # fallback: no fence → use it all
    return code.strip() + "\n"                        # trailing newline = good source hygiene

def write_cpp_file(llm_output, filename="main.cpp"):
    """Clean the model output and write it to a .cpp file. Returns the path."""
    code = extract_cpp(llm_output)
    Path(filename).write_text(code, encoding="utf-8")
    return filename

In [44]:
# now we have enough to send the request to the LLM, next up is to create functions which compile the main.cpp file, run it and give us the output

import subprocess
from pathlib import Path

def compile_and_run(source_file, executable, timeout=120):
    """Compile an existing .cpp file and run it.

    Assumes the file is already written (e.g. by write_cpp_file).
    Returns {"stage", "success", "output", "error"} so the caller can react
    to *where* it failed (compile vs run).
    """
    # Guard: make sure the source file is actually there before we try.
    if not Path(source_file).exists():
        return {"stage": "setup", "success": False, "output": "",
                "error": f"Source file not found: {source_file}"}

    # 1. Compile. stderr is where compile errors go — capture it so we can see
    #    (or later feed back to the model) exactly what failed.
    # compile_cmd = ["clang++", "-std=c++17", "-O3", "-DNDEBUG", source_file, "-o", executable]
    compile_cmd = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
    compiled = subprocess.run(compile_cmd, capture_output=True, text=True)
    if compiled.returncode != 0:
        return {"stage": "compile", "success": False, "output": "", "error": compiled.stderr}

    # 2. Run the executable. "./" tells the shell it's a local file.
    #    timeout guards against a bad port that hangs in an infinite loop.
    try:
        ran = subprocess.run([f"./{executable}"], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        return {"stage": "run", "success": False, "output": "", "error": f"Timed out after {timeout}s"}

    if ran.returncode != 0:
        return {"stage": "run", "success": False, "output": ran.stdout, "error": ran.stderr}

    return {"stage": "done", "success": True, "output": ran.stdout, "error": ""}

In [43]:
# now we create one final function, which takes in the provider name and gives us the output of the c++ code.

def port_and_run_c(provider):
    model, api, url = fetch_model(provider)
    messages = create_message(sys_info=system_info, code_str=pi, compile_info=compile_message)
    client = OpenAI(api_key=api, base_url=url)
    print(f"Now running model: {model}")
    response = client.chat.completions.create(model=model, messages=messages)
    llm_output = response.choices[0].message.content
    extract_cpp(llm_output)
    write_cpp_file(llm_output, filename="main.cpp")
    code_response = compile_and_run("main.cpp", "main")
    if code_response['success']:
        print("****Porting Successful****")
        print(f"C++ Program output: \n{code_response['output']}")
    else:
        print("Porting failed")
        print(f"LLM Response: {llm_output}")


In [17]:
port_and_run_c("claude")

Now running model: claude-fable-5-1
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.381489 seconds



In [16]:
port_and_run_c("gpt")

Now running model: gpt-6-astra
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.387706 seconds



In [18]:
port_and_run_c("sarvam")

Now running model: sarvam-105b


TypeError: expected string or bytes-like object, got 'NoneType'

In [19]:
# sarvam is giving an error, lets check what do we get in the response.
messages = create_message(sys_info=system_info, code_str=pi, compile_info=compile_message)
client = OpenAI(api_key=sarvam_api, base_url=sarvam_baseurl)
response = client.chat.completions.create(model='sarvam-105b', messages=messages)
print(response)

ChatCompletion(id='20260917_2cc97553-9cd4-44d4-b38d-0a27f1752e61', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning_content='We need to convert Python code to high performance C++ code, optimized for Apple M2 with NEON and other extensions. We need to produce C++ code that computes the same result, uses O3, and no debug. The Python code has a function calculate(iterations, param1, param2). Inside, result = 1.0; for i in range(1, iterations+1): compute j = i * param1 - param2; result -= (1/j); j = i * param1 + param2; result += (1/j). Finally returns result. Then main does start_time = time.time(); result = calculate(200,000,000, 4, 1) * 4; end_time = time.time(); print.\n\nWe need to port it. The loop does 200 million iterations, which is heavy. In Python, that would be slow. In C++ with O3, it can be faster but still ma

In [20]:
print(response.choices[0].message.content)

None


In [21]:
print(response.choices[0].message.reasoning_content)

We need to convert Python code to high performance C++ code, optimized for Apple M2 with NEON and other extensions. We need to produce C++ code that computes the same result, uses O3, and no debug. The Python code has a function calculate(iterations, param1, param2). Inside, result = 1.0; for i in range(1, iterations+1): compute j = i * param1 - param2; result -= (1/j); j = i * param1 + param2; result += (1/j). Finally returns result. Then main does start_time = time.time(); result = calculate(200,000,000, 4, 1) * 4; end_time = time.time(); print.

We need to port it. The loop does 200 million iterations, which is heavy. In Python, that would be slow. In C++ with O3, it can be faster but still may be heavy. But we can vectorize with NEON or just let O3 scalar optimization; also can use OpenMP? Not allowed? But we can use OpenMP to parallelize across cores. The constraints: clang++ with OpenMP? Should we include <omp.h> and #pragma omp parallel for reduction(+:result) to speed up.

But 

In [ ]:
# sarvam stopped due to its token limit, lets try by increasing the token limit.

messages = create_message(sys_info=system_info, code_str=pi, compile_info=compile_message)
client = OpenAI(api_key=sarvam_api, base_url=sarvam_baseurl)
response = client.chat.completions.create(model='sarvam-105b', messages=messages, max_tokens=8000)
response

ChatCompletion(id='20260917_6ef77285-fa53-4ab8-af92-faec42717b44', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```cpp\n#include <iostream>\n#include <chrono>\n#include <iomanip>\n\ndouble calculate(long long iterations, double param1, double param2) {\n    double result = 1.0;\n    for (long long i = 1; i <= iterations; ++i) {\n        // This structure is highly amenable to compiler auto-vectorization\n        // via SIMD instructions (like NEON on ARM) at -O3 optimization level.\n        const double term = i * param1;\n        const double j_neg = term - param2;\n        const double j_pos = term + param2;\n        result -= 1.0 / j_neg;\n        result += 1.0 / j_pos;\n    }\n    return result;\n}\n\nint main() {\n    auto start_time = std::chrono::high_resolution_clock::now();\n    \n    const long long iterations = 200000000;\n    const double param1 = 4.0;\n    const double param2 = 1.0;\n    \n    // Set output to fixed-p

In [25]:
# now we got the final complete response, lets try running the code also.

llm_output = response.choices[0].message.content
extract_cpp(llm_output)
write_cpp_file(llm_output, filename="main.cpp")
code_response = compile_and_run("main.cpp", "main")
if code_response['success']:
    print("****Porting Successful****")
    print(f"C++ Program output: \n{code_response['output']}")
else:
    print("Porting failed")
    print(f"LLM Response: {llm_output}")

****Porting Successful****
C++ Program output: 
Result: 3.141593
Execution Time: 0.379737 seconds



In [ ]:
# so we were able to run sarvam's code also, however, the code that was written produced incorrect output.

In [16]:
port_and_run_c("claude")

Now running model: claude-fable-5-1
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.378291 seconds



In [17]:
port_and_run_c("gpt")

Now running model: gpt-6-astra
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.366989 seconds



In [20]:
port_and_run_c("gpt")

Now running model: gpt-6-astra
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.447132 seconds



In [27]:
port_and_run_c("claude")

Now running model: claude-fable-5-1
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.381357 seconds



In [33]:
port_and_run_c("claude")

Now running model: claude-fable-5-1
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.401700 seconds



In [34]:
port_and_run_c("gpt")

Now running model: gpt-6-astra
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.378407 seconds



In [39]:
port_and_run_c("gpt")

Now running model: gpt-6-astra
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.513751 seconds



In [45]:
port_and_run_c("claude")

Now running model: claude-fable-5-1
****Porting Successful****
C++ Program output: 
Result: 3.141592656089
Execution Time: 0.389661 seconds

